# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, with all references by their `@id`.

In [ ]:
# List all record sets and their fields, using @id references
print("Available record sets (by @id):\n")
for record_set in metadata.record_sets:
    print(f"Record set: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields and columns (@id):")
        for field in fields:
            print(f"    Field @id: {field['@id']}  name: {field.get('name', '')}")
            if 'column' in field:
                columns = field['column']
                if isinstance(columns, dict):
                    columns = [columns]
                for column in columns:
                    print(f"      Column @id: {column['@id']}  name: {column.get('name', '')}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# For this dataset, let's extract all available record sets into separate DataFrames
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}, {len(df)} records, {len(df.columns)} columns.")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as ex:
        print(f"Failed to load records for {record_set_id}: {ex}")
        dataframes[record_set_id] = None

# For demonstration, show the head of the first complete loaded record set
sample_record_set = None
for rsid, df in dataframes.items():
    if df is not None and len(df) > 0:
        sample_record_set = rsid
        print(f"Using record set: {rsid}")
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
import numpy as np

if sample_record_set is not None:
    df = dataframes[sample_record_set]
    
    # Try to find a numeric column for demonstration
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        print("No numeric columns found for EDA.")
    else:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize this numeric column
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try a categorical/group field
        group_candidates = df.select_dtypes(include=["object"]).columns.tolist()
        group_field = None
        for col in group_candidates:
            # Ensure it's not too high cardinality
            if df[col].nunique() < min(20, len(df)//5):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group/categorical field found for grouping.")
else:
    print("No data loaded for EDA.")


## 5. Visualization
Visualize a data distribution or field relationship.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_record_set is not None and numeric_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {sample_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have grouping, do a boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} in {sample_record_set}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` package:

- Loaded metadata and record sets
- Inspected available record sets, fields, and columns via their `@id`
- Loaded one record set into a DataFrame and performed basic EDA and visualizations

To go further, adjust the field and record set `@id` variables for your own workflow, and see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) for more advanced schema-driven dataset manipulations.